# Загрузка данных и EDA

Импорты

In [ ]:
!pip install catboost sentence_transformers networkx -q

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
import networkx as nx
from tqdm import tqdm

from transformers import AutoModel, AutoTokenizer
from catboost import CatBoostClassifier
from sentence_transformers import SentenceTransformer

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder


import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import Adam
from torch.nn import CrossEntropyLoss

In [ ]:
items = pd.read_parquet('items.parquet')
train_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')
sample_submission =  pd.read_csv('sample_submission.csv')

In [ ]:
display(items.head())
display(train_data.head())
display(train_data.info())

In [ ]:
print(f"Количество уникальных постов в левом ID: {train_data['leftItemId'].nunique()}. В правом: {train_data['rightItemId'].nunique()}.")
display(set(train_data['target'])) # Все классы таргета
display(train_data.groupby('target').count()) # Количество объектов каждого класса

Посмотрим на пересечения train и test

In [ ]:
unique_ids_train = pd.concat([train_data['leftItemId'], train_data['rightItemId']]).unique()
unique_ids_test = pd.concat([test_data['leftItemId'], test_data['rightItemId']]).unique()

intersect = np.intersect1d(unique_ids_train, unique_ids_test)

print(f"Количество объектов в train: {len(unique_ids_train)}, test: {len(unique_ids_test)}, пересечение: {len(intersect)}")

Как видно 30000 объектов из валидации не присутствуют в тренировочной выборке. Так что графовые фичи не покроют всю валидацию

План работы:
- Сделать бейзлайн на эмбедингах и косинусном расстоянии
- Построить граф связей
- Дообучить трансформер на leftItemId [SEP] rightItemId


# Бейзлайн. Соотношение косинусного расстояния предложений с релевантностью

In [ ]:
# Склеим заголовки и текста чтобы подать их в bert
items['full_text'] = "Заголовок: " + items['title'].astype(str) + ". Содержание: " + items['content'].astype(str)
items['full_text_short'] = items['full_text'].apply(lambda x: " ".join(x.split()[:200]))

# Превращаем в массив с ограничением длины
full_text = items['full_text_short'].values

# Заменим ненужные в сабмите категории
train_data['target'] = train_data['target'].replace({
    'doubles': 'relevant_plus',
    'unavailable': 'no_relevant'
})

In [ ]:
emb_model = SentenceTransformer('cointegrated/rubert-tiny2')

emb_model.max_seq_length = 256
embeddings = emb_model.encode(
    full_text,
    batch_size=64,
    device='cuda',
    show_progress_bar=True
)

id_to_embedding = dict(zip(items['itemId'].tolist(), embeddings))


Получили эмбединги каждого текста по id, теперь из train берем id двух текстов, считаем косинусное сходство и сопоставляем релевантности

##Функция для подсчета косинусного сходства

In [ ]:
def get_cos_sim(data, id_to_emb):
  # Записываем все пары статей
  left = data['leftItemId'].values
  right = data['rightItemId'].values

  zero_vector = np.zeros(312)

  # Получаем эмбеддинги всех пар статей, если такой не стречалось: эмбеддинг = 0
  left_embeddings = np.array([id_to_emb.get(idx, zero_vector) for idx in left])
  right_embeddings = np.array([id_to_emb.get(idx, zero_vector) for idx in right])

  # Делим эмбеддинги на их нормы, для дальнейшего получения косинусного сходства
  left_norm = left_embeddings / np.linalg.norm(left_embeddings, axis=1, keepdims=True)
  right_norm = right_embeddings / np.linalg.norm(right_embeddings, axis=1, keepdims=True)

  cosine_sim = np.sum(left_norm * right_norm, axis=1)

  output = {
      'cosine_sim' : cosine_sim,
      'left_embeddings' : left_embeddings,
      'right_embeddings' : right_embeddings
  }

  return output

In [ ]:
train_data['cosine_sim'] = get_cos_sim(train_data, id_to_embedding)['cosine_sim']

# Получим примерные значения косинусного сходства для каждой категории релевантности
target_sim = pd.DataFrame(train_data.groupby('target', as_index=False)['cosine_sim'].mean())
display(target_sim)

Получили значения косинусного сходства для векторов разной релевантности. Теперь можно применить его к тесту

## Обучим решающее дерево на косинусном сходстве

In [ ]:
X_tree = train_data[['cosine_sim']]
y_tree = train_data['target']

X_train_tree, X_test_tree, y_train_tree, y_test_tree = train_test_split(
    X_tree,
    y_tree,
    test_size=0.2,
    random_state=42,
    stratify=y_tree)

tree = DecisionTreeClassifier(max_depth=4, random_state=42)
tree.fit(X_train_tree, y_train_tree)

y_pred_tree = tree.predict(X_test_tree)
print(classification_report(y_test_tree, y_pred_tree))


Результаты не впечатляющие. Попробуем послать бейзлайн на лидерборд

In [ ]:
X_val_tree = pd.DataFrame(get_cos_sim(test_data, id_to_embedding)['cosine_sim'], columns=['cosine_sim'])

y_val_tree = tree.predict(X_val_tree)

In [ ]:
submission1 = pd.DataFrame(
    {
        "Unnamed: 0": samle_submission['Unnamed: 0'],
        'target' : y_val_tree
    }
)

submission1.to_csv('submission1.csv', index=False)

Скор 0.42 на метрике Weighted F1.

## Обучим catboost на косинусном сходстве

In [ ]:
X_catboost1 = train_data[['cosine_sim']]
y_catboost1 = train_data['target']

X_train_catboost1, X_test_catboost1, y_train_catboost1, y_test_catboost1 = train_test_split(
                                                                                            X_catboost1,
                                                                                            y_catboost1,
                                                                                            test_size=0.2,
                                                                                            random_state=42,
                                                                                            stratify = y_catboost1)

catboost1 = CatBoostClassifier(iterations=1000, depth=4, early_stopping_rounds=50)

catboost1.fit(X_train_catboost1, y_train_catboost1)
y_pred_catboost1 = catboost1.predict(X_test_catboost1)


In [ ]:
print(f"CatBoost: {classification_report(y_test_catboost1, y_pred_catboost1)}")


print(f"Desicion tree: {classification_report(y_test_tree, y_pred_tree)}")

In [ ]:
X_val_catboost1 = pd.DataFrame(get_cos_sim(test_data, id_to_embedding)['cosine_sim'], columns=['cosine_sim'])
y_val_catboost1 = catboost1.predict(X_val_catboost1)

submission2 = pd.DataFrame(
    {
        "Unnamed: 0": samle_submission['Unnamed: 0'],
        'target' : y_val_catboost1.squeeze()
    }
)

submission2.to_csv('submission2.csv', index=False)

С CatBoost скор поднялся до 0.43.

## Добавим больше фичей, основаных на схожести векторов эмбедингов

In [ ]:
def add_features(left, right):
  features = {}
  # Евклидово расстояние
  features['norm'] = np.linalg.norm(left - right, axis=1)
  #Косинусное сходство
  left_norm = left / np.linalg.norm(left, axis=1, keepdims=True)
  right_norm = right / np.linalg.norm(right, axis=1, keepdims=True)
  features['cos_norm'] = np.sum(left_norm * right_norm, axis=1)

  #Покоординатное скалярное произведение (произведение Адамара)
  hadamard = left * right
  for i in range(hadamard.shape[1]):
    features[f'hadamar_{i}'] = hadamard[:, i]

  return pd.DataFrame(features)

In [ ]:
catboost2 = CatBoostClassifier(iterations=1000, early_stopping_rounds=50, task_type='GPU')

left_embeddings_train_catboost2 = get_cos_sim(train_data, id_to_embedding)['left_embeddings']
right_embeddings_train_catboost2 = get_cos_sim(train_data, id_to_embedding)['right_embeddings']

X_catboost2 = add_features(left_embeddings_train_catboost2,
                           right_embeddings_train_catboost2)

y_catboost2 = train_data['target']

X_train_catboost2, X_test_catboost2, y_train_catboost2, y_test_catboost2 = train_test_split(X_catboost2, y_catboost2, test_size=0.3, random_state=42)

catboost2.fit(X_train_catboost2, y_train_catboost2)

y_pred_catboost2 = catboost2.predict(X_test_catboost2)



In [ ]:
print(classification_report(y_test_catboost2, y_pred_catboost2))

In [ ]:
left_embeddings_val_catboot2 = get_cos_sim(test_data, id_to_embedding)['left_embeddings']
right_embeddings_val_catboost2 = get_cos_sim(test_data, id_to_embedding)['right_embeddings']

X_val_catboost2 = add_features(left_embeddings_val_catboot2, right_embeddings_val_catboost2)

y_val_catboost2 = catboost2.predict(X_val_catboost2)

submission3 = pd.DataFrame(
    {
        "Unnamed: 0": samle_submission['Unnamed: 0'],
        'target' :y_val_catboost2 .squeeze()
    }
)

submission3.to_csv('sample_submission3.csv', index=False)

Добавление новых фичей повысило скор до 0.48. Можно лучше

# Использование графовых фичей

In [ ]:
def build_graph(train_df, test_df, items_df, text_embeddings, threshold=0.75):
    G = nx.Graph()

    # Собираем все уникальные пары левыйID-правыйID
    all_pairs = pd.concat([
        train_df[['leftItemId', 'rightItemId']],
        test_df[['leftItemId', 'rightItemId']]
    ]).drop_duplicates()

    # Вершины - все уникальные ID
    unique_items = set(all_pairs['leftItemId']).union(set(all_pairs['rightItemId']))
    G.add_nodes_from(unique_items)

    # Словарь для получения индекса эмбединга по индексу текста
    item_id_to_idx = {item_id: idx for idx, item_id in enumerate(items_df['itemId'])}

    # Берем ID для которых есть эмбеддинги
    valid_mask = all_pairs['leftItemId'].isin(item_id_to_idx) & all_pairs['rightItemId'].isin(item_id_to_idx)
    pairs_filtered = all_pairs[valid_mask].copy()

    # Переводим ID товаров в индексы векторов
    left_indices = pairs_filtered['leftItemId'].map(item_id_to_idx).values
    right_indices = pairs_filtered['rightItemId'].map(item_id_to_idx).values

    #  Извлекаем матрицы эмбеддингов для левой и правой сторон
    # Матрицы будут размера (размер_датасета x размер_эмбеддинга)
    left_embs = text_embeddings[left_indices]
    right_embs = text_embeddings[right_indices]


    dot_product = np.sum(left_embs * right_embs, axis=1)

    # Считаем длины векторов (нормы)
    left_norms = np.linalg.norm(left_embs, axis=1)
    right_norms = np.linalg.norm(right_embs, axis=1)

    # Итоговое сходство для каждой пары
    # Добавляем 1e-9 в знаменатель, чтобы защититься от деления на ноль, если есть пустые векторы
    scores = dot_product / (left_norms * right_norms + 1e-9)

    # 5. Фильтруем по порогу и создаем ребра
    print("Фильтрация ребер по порогу...")
    edge_mask = scores >= threshold

    # Достаем ID товаров, которые прошли порог
    edges_u = pairs_filtered['leftItemId'].values[edge_mask]
    edges_v = pairs_filtered['rightItemId'].values[edge_mask]

    # Объединяем их в формат, понятный NetworkX (список кортежей)
    edges_to_add = list(zip(edges_u, edges_v))

    G.add_edges_from(edges_to_add)
    print(f"Итоговый граф: {G.number_of_nodes()} вершин и {G.number_of_edges()} ребер.")
    return G

In [ ]:
# Ошибочный вариант функции, из-за которого произошла утечка данных, но благодаря которому удалось получить одно из лучших решений на приватном лидерборде.(см вывод)

"""def build_graph(train_df):
    pos_pairs = train_df[train_df['target'] == 'relevant'] # Будем оценивать только релевантные товары
    G = nx.Graph()
    edges = zip(pos_pairs['leftItemId'], pos_pairs['rightItemId']) # Строим уже известные связи
    G.add_edges_from(edges)
    return G # В итоге получился граф с группами связанных объектов"""

def get_graph_features(df, G):
    degrees = dict(G.degree()) # Словарь с количеством связей каждого объекта
    pagerank = nx.pagerank(G) # Словарь с количеством ссылающихся объектов на данный

    feats = {
        'l_deg': [], 'r_deg': [], 'l_pr': [], 'r_pr': [],
        'common_n': [], 'jaccard': [], 'adamic_adar': []
    }

    for _, row in df.iterrows():
        u, v = row['leftItemId'], row['rightItemId']

        # Составление словаря с фичами: степень узла и PageRank
        feats['l_deg'].append(degrees.get(u, 0))
        feats['r_deg'].append(degrees.get(v, 0))
        feats['l_pr'].append(pagerank.get(u, 0))
        feats['r_pr'].append(pagerank.get(v, 0))

        if G.has_node(u) and G.has_node(v):
            feats['common_n'].append(len(list(nx.common_neighbors(G, u, v))))
            feats['jaccard'].append(list(nx.jaccard_coefficient(G, [(u, v)]))[0][2])
            # Используем срез потому что функция является генератором, оборачиваем ее в список в котором лежит: [(u, v, jaccart_c)]
            try: # В теории возможно что у соседа u и v будет одна связь, тогда log знаменателе меры Адамик-Адара ставит равен 0(к примеру u и v - один и тот же объект)
                feats['adamic_adar'].append(list(nx.adamic_adar_index(G, [(u, v)]))[0][2])
            except ZeroDivisionError:
                feats['adamic_adar'].append(0)
        else:
            feats['common_n'].append(0)
            feats['jaccard'].append(0)
            feats['adamic_adar'].append(0)

    return pd.DataFrame(feats)

In [ ]:
graph = build_graph(train_data, test_data, items, embeddings) # Построим граф

graph_data = get_graph_features(train_data, graph) # Получим фичи для обучения
graph_data.describe()

In [ ]:
X_catboost3 = pd.concat([X_catboost2, graph_data], axis=1)
X_catboost3.fillna(0)

In [ ]:

X_train_catboost3, X_test_catboost3, y_train_catboost3, y_test_catboost3 = train_test_split(X_catboost3,
                                                                                            train_data['target'],
                                                                                            test_size=0.35,
                                                                                            random_state=42,
                                                                                            stratify=train_data['target'])

model = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='TotalF1',
    task_type='GPU',
    verbose=False

)

#  Задаем сетку параметров для перебора
param_grid = {
    'depth': [4, 6, 8],
    'l2_leaf_reg': [1, 3, 5, 7],
    'learning_rate': [0.03, 0.1, 0.2]
}



grid_search_result = model.grid_search(
    param_grid,
    X=X_train_catboost3,
    y=y_train_catboost3,
    cv=3
    partition_random_seed=42,
    plot=True
)

In [ ]:


catboost3 = CatBoostClassifier(
    iterations=20000,
    task_type='GPU',
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=5
)

catboost3.fit(
    X_train_catboost3, y_train_catboost3

)

y_pred_catboost3 = catboost3.predict(X_test_catboost3)

In [ ]:
grid_search_result['params']

In [ ]:
print(classification_report(y_test_catboost3, y_pred_catboost3))

In [ ]:
val_graph_catboost3 = get_graph_features(test_data, graph)
val_feat_catboost3 = add_features(left_embeddings_val_catboot2 , right_embeddings_val_catboost2)
X_val_catboost3 = pd.concat([val_graph_catboost3, val_feat_catboost3], axis=1)
y_val_catboost3 = catboost3.predict(X_val_catboost3)

submission4 = pd.DataFrame(
    {
        "Unnamed: 0": samle_submission['Unnamed: 0'],
        'target' : y_val_catboost3.squeeze()
    }
)

submission4.to_csv('submission4.csv', index=False)

Со всеми добавленными фичами лучший скор: 0.49. У первого места в данном соревновании 0.54

Попробуем дообучить обученный трансформер на классификацию релевантности текстов

# Дообучение предобученного трансформера

In [ ]:

# Класс датасета для DataLoader
class DataSet():
  def __init__(self, df, tokenizer, items_df, max_length=320):
    self.df = df
    self.tokenizer = tokenizer
    self.max_length = max_length
    self.items_df = items_df

    self.id_to_text = dict(zip(items_df['itemId'], items_df['full_text_short'].astype(str)))

  def __len__(self):
    return len(self.df)

  def __getitem__(self, idx):
    row = self.df.iloc[idx]
    tokenized = self.tokenizer(
        self.id_to_text.get(row['leftItemId'], ""),
        self.id_to_text.get(row['rightItemId'], ""),
        padding='max_length',
        truncation=True,
        max_length=self.max_length,
        return_attention_mask=True,
        return_token_type_ids=True,
        return_tensors='pt')

    items = {key: val.squeeze(0) for key, val in tokenized.items()}
    if 'target' in row:
      items['labels'] = torch.tensor((row['target']), dtype=torch.long)

    return items


In [ ]:


# Численно закодируем target и уберем doubles, unavailable
le = LabelEncoder()
le.fit(train_data['target'])
train_data['target'] = le.transform(train_data['target'])

In [ ]:

# Класс предобученного трансформера и классфикатора.
class Model(nn.Module):
  def __init__(self, path='ai-forever/ruBert-base'):
    super().__init__()
    self.backbone = AutoModel.from_pretrained(path)

    hidden_size = self.backbone.config.hidden_size

    self.classes = nn.Sequential(
          nn.Linear(hidden_size, hidden_size),
          nn.LayerNorm(hidden_size),
          nn.ReLU(),
          nn.Linear(hidden_size, 4),

    )

  def forward(self, input_ids, attention_mask, token_type_ids):

    output = self.backbone(
        input_ids=input_ids,
        attention_mask=attention_mask,
        token_type_ids=token_type_ids
    )

    logits = self.classes(output.last_hidden_state[:, 0, :])

    return logits


In [ ]:


tokenizer = AutoTokenizer.from_pretrained('ai-forever/ruBert-base')

train_sub, _ = train_test_split(train_data, train_size=40000, stratify=train_data['target'], random_state=42)
dataset_train = DataSet(train_sub, tokenizer, items)

data_loader_train = DataLoader(dataset_train, batch_size=32,
    shuffle=True,
    pin_memory=True,
    num_workers=2)

dataset_test = DataSet(test_data, tokenizer, items)
data_loader_test = DataLoader(
    dataset_test,
    batch_size=32,
    shuffle=False,
    pin_memory=False,
    num_workers=2
)

In [ ]:
class_weights = (train_data['target'].value_counts(normalize=True)).sort_index()
class_weights_tensor = torch.tensor(class_weights.values, dtype=torch.float32).to('cuda')
class_weights_tensor

In [ ]:
bert = Model()

bert.to('cuda')
optimizer = Adam(bert.parameters(), lr=2e-5)

criterion = CrossEntropyLoss(weight=class_weights_tensor)

Обучим классификатор кастомного класса модели

In [ ]:


global_loss = []
n_epochs = 2

bert.train()

for epoch in range(n_epochs):
  epoch_losses = []
  for batch in tqdm(data_loader_train):
    optimizer.zero_grad()

    batch = {k: v.to('cuda') for k, v in batch.items()}
    labels = batch.pop('labels')

    outputs = bert(**batch)
    loss = criterion(outputs, labels.squeeze(-1))

    loss.backward()
    optimizer.step()
    epoch_losses.append(loss.item())
  global_loss.append(sum(epoch_losses) / len(epoch_losses))


print(global_loss)


In [ ]:
bert.eval()
outputs = []

with torch.no_grad():
  for batch in tqdm(data_loader_test):

    batch = {k: v.to('cuda') for k, v in batch.items()}
    logits = bert(**batch)
    output = torch.argmax(logits, axis=-1).cpu().numpy()

    outputs.extend(output)

y_val = le.inverse_transform(outputs)


In [ ]:
submission5 = pd.DataFrame(
    {
        "Unnamed: 0": samle_submission['Unnamed: 0'],
        'target' :y_val.squeeze()
    }
)

submission5.to_csv('submission5.csv', index=False)

# Вывод


- Получилось войти в топ 11 лучших решений данного соревнования. Лучший результат получился с подобранными параметрами catboost с помощью grid_search и графовыми фичами. Предобученый трансформер не дал хороших результатов. Скорее всего это произошло из-за нечеткой границы между классами (таблица среднего косинусного сходства). Разница между relevant и relevant_minus равна двум сотым. В то время как графовые признаки просто сгрупировали похожие тексты.



- В ячейке с функцией графовых признаков присутствует ошибочная функция построения графов, обеспечивающая утечку информаци:
- Ошибка заключалась в том что для построения графа брались только релевантные объекты из train. Поэтому на тесте, который является подмножеством train, модель просто знала какие объекты имеют класс relevant, которых 72000 из 211000.
